# A binary X, and running regressions in Python: practice notebook

**Causal and Predictive Analytics · Asynchronous module** · Simon Business School · Prof. Tianli Xia

All the Python from the module's web page, plus short exercises (cells marked `# YOUR CODE HERE`). Five small data sets:

| File | What it is |
|---|---|
| `cps12.csv` | 3,955 full-time workers aged 25–34 with a college degree (Current Population Survey, 2012): hourly `earnings`, `male`, `female` |
| `resume.csv` | 4,870 résumés from a hiring experiment: `callback` (1 = the employer called back), `white_name` (1 = a White-sounding name, assigned at random) |
| `hprices.csv` | 88 houses: `price` (in $1,000s) and size `sqrft` |
| `diamonds.csv` | 380 diamonds: `price`, `carats`, `color` (1 = colorless or near-colorless) |
| `wines.csv` | 230 wines: `Price`, rating `Score`, `Year` |

**In Colab:** *File → Save a copy in Drive* first, so your changes are kept. Run cells with **Shift+Enter**.

## 0. Setup and data

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

pd.options.display.float_format = "{:,.4f}".format

In [ ]:
from pathlib import Path

SITE = "https://tianlixia.com/causal-analytics/GBA472/A3/"   # the module's web address; filled in when the notebook is published, so the data load in Colab


def load(name):
    """Read a data file from the folder next to this notebook, or from the module's website."""
    for folder in [Path("data"), Path("web/data"), Path("../web/data")]:
        if (folder / name).exists():
            return pd.read_csv(folder / name)
    return pd.read_csv(SITE + "data/" + name)


cps = load("cps12.csv")
resume = load("resume.csv")
housedata = load("hprices.csv")
diamonds = load("diamonds.csv")
winedata = load("wines.csv")
print(cps.shape, resume.shape, housedata.shape, diamonds.shape, winedata.shape)
cps.head()

## 1. A binary X: the wage gap

### 1.1 The difference in means, by hand

The table in the module: mean, standard deviation and count of earnings for men and women.

In [ ]:
g = cps.groupby("male").earnings.agg(["mean", "std", "count"])
g

The standard error of the difference between two independent sample means is
$SE(\bar Y_m-\bar Y_w)=\sqrt{s_m^2/n_m+s_w^2/n_w}$.

In [ ]:
ym, sm_, nm = g.loc[1]          # men
yw, sw, nw = g.loc[0]           # women
gap = ym - yw
se = np.sqrt(sm_**2 / nm + sw**2 / nw)
t = gap / se
p = 2 * stats.norm.cdf(-abs(t))
print(f"gap = {gap:.3f}, SE = {se:.3f}, t = {t:.2f}, p-value = {p:.2g}")
print(f"95% CI: ({gap - 1.96 * se:.2f}, {gap + 1.96 * se:.2f})")

**Exercise 1.** Compute the 99% confidence interval for the gap (critical value 2.58). Does it contain zero?

In [ ]:
# YOUR CODE HERE


### 1.2 The same thing as a regression

$\text{earnings}_i=\beta_0+\beta_1\,\text{male}_i+e_i$: $\hat\beta_0$ is the women's mean, $\hat\beta_0+\hat\beta_1$ the men's, $\hat\beta_1$ the gap.

In [ ]:
est = smf.ols("earnings ~ male", data=cps).fit()
print(est.summary().tables[1])
print("intercept + coefficient =", round(est.params.sum(), 4), "  men's mean =", round(ym, 4))

The regression's standard error (0.353) differs from the formula above (0.352) in the third decimal: the regression's default pools the two groups' variances. With heteroskedasticity-robust standard errors the two agree:

In [ ]:
robust = smf.ols("earnings ~ male", data=cps).fit(cov_type="HC1")
print("default SE:", round(est.bse["male"], 4), "  robust SE:", round(robust.bse["male"], 4), "  formula:", round(se, 4))

**Exercise 2.** Regress earnings on the `female` dummy instead. Before you run it, predict the intercept and the coefficient.

In [ ]:
# YOUR CODE HERE


**Exercise 3.** Why not use both dummies? Check that `male + female` equals 1 for every worker, the same as the column of ones behind the intercept.

In [ ]:
# YOUR CODE HERE


## 2. A treatment dummy: the hiring experiment

Résumés got a White-sounding or an African-American-sounding name **at random**. With random assignment, the coefficient on the treatment dummy estimates the causal effect of the name.

In [ ]:
print(resume.groupby("white_name").callback.agg(["mean", "count"]))
est_r = smf.ols("callback ~ white_name", data=resume).fit()
print(est_r.summary().tables[1])

**Exercise 4.** By what percentage does a White-sounding name raise the callback rate, relative to the rate for African-American-sounding names? Use `est_r.params`.

In [ ]:
# YOUR CODE HERE


## 3. OLS in Python: the house prices

### 3.1 Way 1: the matrix way (`sm.OLS`; add the constant yourself)

In [ ]:
y = housedata[["price"]]
X = housedata[["sqrft"]]
X = sm.add_constant(X)        # a column of 1s, for beta0
est = sm.OLS(y, X).fit()
print(est.summary())

### 3.2 Way 2: the formula way (`smf.ols`; the intercept is automatic)

In [ ]:
est = smf.ols("price ~ sqrft", data=housedata).fit()
print(est.summary())

### 3.3 The rest of Excel's output

In [ ]:
print(sm.stats.anova_lm(est, typ=2))          # ESS (sqrft row) and SSR (Residual row), the F-test
print("TSS =", round(est.ess + est.ssr, 1))
print("MSE =", round(est.scale, 1), "  SER =", round(np.sqrt(est.scale), 2))

### 3.4 Pulling numbers out

In [ ]:
print(est.params)
print(est.bse)
print(est.conf_int())
print("R2 =", round(est.rsquared, 3), "  n =", int(est.nobs))
print(est.predict(pd.DataFrame({"sqrft": [2000]})))     # predicted price of a 2,000 sq ft house, in $1,000s

**Exercise 5.** One more square foot goes with how many dollars more on the price? (Careful with the units.) Compute it from `est.params`, with its 95% confidence interval in dollars.

In [ ]:
# YOUR CODE HERE


## 4. Two more data sets

### 4.1 The diamonds

In [ ]:
est = smf.ols("price ~ carats", data=diamonds).fit()
print(est.summary().tables[1])
print("R-squared:", round(est.rsquared, 3), "  n =", int(est.nobs))

### 4.2 Everything you need, in one block (the wines)

Copy this block into any notebook, change the file name and the variable names.

In [ ]:
# pandas is used to load the data
import pandas as pd
# statsmodels.formula.api is used to run the regression
import statsmodels.formula.api as smf
# read in the data and store it in a data frame called winedata
winedata = load("wines.csv")          # or pd.read_csv("<web address or file name>")
# run the regression and store the results in est
est = smf.ols("Price ~ Score", data=winedata).fit()
# summarize and print the results
print(est.summary())

**Exercise 6.** `color` in the diamonds file is a dummy (1 = colorless or near-colorless). Regress `price` on `color`. What do the intercept and the coefficient mean, in words? Is the difference statistically significant?

In [ ]:
# YOUR CODE HERE


**Exercise 7.** Is that $650 the effect of color on price? Compare the average size (`carats`) of the two groups. What does that suggest?

In [ ]:
# YOUR CODE HERE


---
*That's all. Next: the assumptions under which these standard errors and tests are right, and prediction with confidence and prediction intervals.*